# Research 2 — SECOND dataset preparation (R2-P05-T03, D-011)

Purpose: one reproducible, documented data foundation for Research 2 (SECOND, 6 land-cover classes,
bi-temporal 512×512 RGB tiles with per-time semantic labels). This notebook **installs/inspects** the
dataset, documents its structure, **verifies the frozen split manifest** (D-008/D-009) and reports
leakage diagnostics. It trains nothing and never reads test labels for any decision (D-007).

How to use: all expensive or state-changing steps sit behind flags that default to `False`. With the
defaults, the notebook is read-only (installation check, manifest verification, audit-report summary).
Nothing below is pre-executed: no output is stored in this file, so every number you see comes from your
own run on your own `data/SECOND`.

Related decisions (`docs/research2/DECISION_LOG.md`): D-007 official splits and untouched test;
D-008 validation = 15 % of official-train *groups* (seed 2099); D-009 official test used as is;
D-018 no baked-in augmentation.

In [ ]:
import hashlib
import importlib.util
import json
from pathlib import Path

import numpy as np
from PIL import Image

from satchange.data.second_binary import verify_second_sources
from satchange.semantic.second import (
    SECOND_CLASSES, SECOND_COLORS, bootstrap_second, decode_label, discover_second,
    prepare_second_splits, read_second_split_manifest,
)

# ---- Flags: everything that downloads, writes, or is expensive defaults to False -------------
ALLOW_DOWNLOAD = False       # download the ~3.8 GB release (never done silently)
RUN_INSTALL = False          # call bootstrap_second (ZIP given in ARCHIVE or the download above)
RUN_PREPARE_SPLITS = False   # write the split manifest (an existing manifest is NEVER replaced)
RUN_AUDIT = False            # leakage/duplicate audit (hashes + compares every tile; minutes)
RUN_VERIFY_HASHES = False    # re-hash every file listed in the manifest (minutes)
RUN_VISUALIZE = False        # show a few tiles with decoded labels

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
SECOND_ROOT = PROJECT_ROOT / "data" / "SECOND"
MANIFEST = SECOND_ROOT / "second_split_manifest.json"
AUDIT_JSON = PROJECT_ROOT / "docs" / "research2" / "second_audit.json"
ARCHIVE = None               # e.g. PROJECT_ROOT / "data" / "SECOND-<hash>.zip"
SPLIT_SEED, VAL_FRACTION = 2099, 0.15
print("project root:", PROJECT_ROOT)
print("SECOND root :", SECOND_ROOT, "(exists)" if SECOND_ROOT.exists() else "(missing)")

## 1. Source and installation

Source: https://captain-whu.github.io/SCD/ (SECOND, Yang et al.). The release observed during Research 2
setup is **nested**: `SECOND_total_test.zip` (→ `test/`) and `SECOND_train_set.rar` (→ `train/`), each with
`im1`, `im2`, `label1`, `label2`. `bootstrap_second` does not unpack that nesting yet (known gap in
`PROGRESS.md`); extract the nested archives manually into `data/SECOND/{train,test}` if you install from scratch.
Checksums recorded in `data/SECOND/source.json` are **local** SHA-256 values, not author-published ones.

In [ ]:
if RUN_INSTALL:
    bootstrap_second(SECOND_ROOT, archive=ARCHIVE, allow_download=ALLOW_DOWNLOAD)

try:
    layout = discover_second(SECOND_ROOT)
except (FileNotFoundError, ValueError) as error:
    layout = None
    print("SECOND layout is not usable:", error)
if layout is not None:
    for split in ("train", "val", "test"):
        print(f"{split:5s}: {len(layout.get(split, []))} labelled pairs"
              + ("" if split in layout else "  (directory not present)"))
source_json = SECOND_ROOT / "source.json"
if source_json.is_file():
    print(json.dumps(json.loads(source_json.read_text(encoding="utf-8")), indent=1)[:1500])

## 2. Classes and label decoding

Labels are RGB colour maps. `decode_label` converts them to canonical local IDs independent of any PNG
palette: **0 = no change (white)**, 1–6 = the six land-cover classes. Unknown colours raise an error
(no implicit fallback). A tile pixel is *changed* when its time-1 label is not 0; the annotation protocol
requires the time-1 and time-2 change supports to coincide, which `SECONDBinaryDataset` checks per sample.

In [ ]:
print("canonical ID -> class")
print("  0 -> no change")
for index, name in enumerate(SECOND_CLASSES, start=1):
    colour = next(c for c, i in SECOND_COLORS.items() if i == index)
    print(f"  {index} -> {name}  (RGB {colour})")

## 3. Frozen split manifest (D-008, D-009)

The official release has a labelled train (2968 pairs) and a labelled test (1694 pairs) and **no
validation split**. Validation is carved once from official-train *groups* (whole groups, seed 2099,
15 %), keeping the few within-train near-duplicate/adjacent tile groups together; the official test is used
unchanged and stays untouched until the frozen final evaluation. An existing manifest is never replaced —
the writer refuses different content — so the guarded cell below is idempotent for the frozen manifest and
requires the audit report (section 4) for the group merging.

In [ ]:
if RUN_PREPARE_SPLITS:
    audit = json.loads(AUDIT_JSON.read_text(encoding="utf-8"))
    spec = importlib.util.spec_from_file_location(
        "second_audit", PROJECT_ROOT / "src" / "scripts" / "second_audit.py")
    second_audit = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(second_audit)
    layout_for_groups = discover_second(SECOND_ROOT)
    sample_ids = [row["sample_id"] for rows in layout_for_groups.values() for row in rows]
    prepare_second_splits(
        SECOND_ROOT, MANIFEST, split_seed=SPLIT_SEED, val_fraction=VAL_FRACTION,
        source_groups=second_audit.source_groups_from_report(audit, sample_ids))
    print("manifest written or identical content confirmed:", MANIFEST)

In [ ]:
manifest = read_second_split_manifest(MANIFEST)   # also verifies the content digest
splits = {name: set(ids) for name, ids in manifest["splits"].items()}
print("manifest_sha256      :", manifest["manifest_sha256"])
print("split policy / seed  :", manifest["policy"], manifest["split_seed"])
print("counts               :", {k: len(v) for k, v in splits.items()})
print("official counts      :", manifest["official_counts"])
print("geographic independence verified:", manifest["geographic_independence_verified"],
      "| grouping basis:", manifest["group_basis"])
names = list(splits)
overlap = {(a, b): len(splits[a] & splits[b]) for i, a in enumerate(names) for b in names[i + 1:]}
print("pairwise overlaps    :", overlap)
assert all(count == 0 for count in overlap.values()), "splits overlap"
if layout is not None and layout.get("test"):
    official_test = {row["sample_id"] for row in layout["test"]}
    print("test == official test:", splits["test"] == official_test)
else:
    print("official test directory is not extracted on this machine; test equality not checked here.")
if RUN_VERIFY_HASHES:
    print("files re-hashed OK  :", verify_second_sources(SECOND_ROOT))

## 4. Diagnostics: leakage and basic statistics

`src/scripts/second_audit.py` (read-only) hashes and compares every tile and writes
`docs/research2/second_audit.json`: label sanity (unknown colours), exact duplicates, near-duplicate pairs,
adjacent-edge candidates, class and transition shares, changed-pixel fraction, tiles without change.
Official IDs interleave train and test (random tile assignment, no geographic metadata), so geographic
independence **cannot be verified**; cross-split candidates are reported, not repaired (official splits stay).

In [ ]:
if RUN_AUDIT:
    spec = importlib.util.spec_from_file_location(
        "second_audit", PROJECT_ROOT / "src" / "scripts" / "second_audit.py")
    second_audit = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(second_audit)
    report = second_audit.audit(SECOND_ROOT)
    report.pop("split_of_sample", None)
    AUDIT_JSON.parent.mkdir(parents=True, exist_ok=True)
    AUDIT_JSON.write_text(json.dumps(report, indent=1, sort_keys=True) + "\n", encoding="utf-8")

if AUDIT_JSON.is_file():
    audit = json.loads(AUDIT_JSON.read_text(encoding="utf-8"))
    print("exact duplicate groups      :", audit["exact_duplicate_groups"])
    print("near-duplicate/edge, train  :", len(audit["candidates_within_train"]))
    print("near-duplicate/edge, cross  :", len(audit["candidates_across_splits"]))
    print("merged groups kept together :", len(audit["merged_groups"]))
    print(json.dumps(audit["summary"], indent=1)[:3000])
else:
    print("No audit report yet; set RUN_AUDIT = True to create", AUDIT_JSON)

In [ ]:
if RUN_VISUALIZE:
    import matplotlib.pyplot as plt
    rows = layout["train"][:3]
    figure, axes = plt.subplots(len(rows), 4, figsize=(12, 3 * len(rows)))
    for axis_row, row in zip(axes, rows, strict=True):
        images = [Image.open(row["before"]), Image.open(row["after"])]
        labels = []
        for key in ("before_semantic", "after_semantic"):
            ids = decode_label(row[key])
            palette = np.zeros((*ids.shape, 3), np.uint8)
            for colour, identifier in SECOND_COLORS.items():
                palette[ids == identifier] = colour
            labels.append(palette)
        for axis, image in zip(axis_row, [*images, *labels], strict=True):
            axis.imshow(image)
            axis.axis("off")
        axis_row[0].set_title(row["sample_id"], fontsize=8)
    plt.tight_layout()
    plt.show()

## 5. Preprocessing and augmentation policy

* **Mandatory preprocessing only:** exact label decoding to canonical IDs, RGB scaled to [0, 1] float32,
  native 512×512 (no resizing unless a model requires it; then it is model-specific and documented in that
  experiment). No mean/std normalisation, matching the LEVIR-CD loaders.
* **Data preparation applies no augmentation and bakes nothing into files.** Training-time synchronized
  flips/rot90 (`RandomFlipRotate`, train only, explicit seeds) are an experiment-level choice recorded as
  provisional D-018 (D-034); no artificial noise; never on validation or test.
* The split manifest, class mapping and preprocessing above are frozen before any confirmatory experiment;
  all models use the same manifest.